# 05 — Qwen2.5-VL-3B-Instruct + LoRA

Affinage LoRA du **modèle de langue uniquement** (encodeur visuel gelé), sur les 3 392 lignes du train.

| `ECHELLE` | Images | Lots | CER test |
|---|---|---|---|
| 1 | taille d'origine | 2 lignes × 8 pas d'accumulation | 22,1 % |
| 2 | agrandies ×2 (plafond 1 M de pixels) | 1 ligne × 16 pas d'accumulation | 30,2 % |

Dans les deux cas : 16 lignes par mise à jour, 2 époques, taux d'apprentissage 2e-4 (cosinus), LoRA r = 16, α = 32.

**Environnement** : **Kaggle**, Accelerator *GPU T4 x2*, Internet activé, dataset `lignes-occitan-poly` (le dossier produit par le notebook 02) ajouté en Input. À lancer avec **Save Version → Save & Run All** : l'entraînement dure environ 2 h (`ECHELLE = 1`) ou 6 h (`ECHELLE = 2`).

In [ ]:
ECHELLE = 2       # 1 ou 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                            # un seul GPU : pas de DataParallel
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # limite la fragmentation mémoire

!pip uninstall -y -q torchao          # version préinstallée incompatible avec peft
!pip install -q peft jiwer

In [ ]:
import json, csv, time, unicodedata, torch
from pathlib import Path
from collections import defaultdict
from PIL import Image

DATA = next(Path("/kaggle/input").rglob("lignes.jsonl")).parent
tout = [json.loads(l) for l in open(DATA / "lignes.jsonl", encoding="utf-8")]
splits = {s: [e for e in tout if e["split"] == s] for s in ("train", "dev", "test")}
print({s: len(v) for s, v in splits.items()})

# Sur Kaggle, les noms de fichiers accentués sont stockés en NFD : on les indexe en NFC
INDEX = {unicodedata.normalize("NFC", p.name): p for p in (DATA / "images").iterdir()}
def chemin_image(e):
    return INDEX[unicodedata.normalize("NFC", Path(e["image"]).name)]
assert all(unicodedata.normalize("NFC", Path(e["image"]).name) in INDEX for e in tout)

SUFFIXE = "" if ECHELLE == 1 else f"_x{ECHELLE}"
OUT = Path("/kaggle/working")

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

MODELE = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(MODELE)
model = AutoModelForImageTextToText.from_pretrained(MODELE, dtype=torch.float16, device_map="cuda:0")
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
))
model.print_trainable_parameters()   # ≈ 29,9 M paramètres, 0,79 %

## Préparation des exemples

Chaque exemple est une conversation : l'utilisateur envoie l'image et la consigne, l'assistant répond par la transcription. Dans les `labels`, tout ce qui précède la réponse vaut `-100` : le modèle n'apprend qu'à produire la transcription et le token de fin `<|im_end|>`.

In [ ]:
MAX_PIXELS = 1_000_000   # plafond : sans lui, les plus grandes lignes agrandies dépassent la mémoire du T4

def agrandir(img):
    facteur = max(1.0, min(ECHELLE, (MAX_PIXELS / (img.width * img.height)) ** 0.5))
    if facteur == 1.0:
        return img
    return img.resize((round(img.width * facteur), round(img.height * facteur)), Image.LANCZOS)

CONSIGNE = (
    "Transcris exactement cette ligne d'un manuscrit médiéval en occitan. "
    "Garde les abréviations telles qu'elles sont écrites, sans les développer. "
    "Réponds uniquement par la transcription."
)

def preparer(e):
    image = agrandir(Image.open(chemin_image(e)).convert("RGB"))
    question = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": CONSIGNE},
    ]}]
    complet = question + [{"role": "assistant", "content": [{"type": "text", "text": e["text"]}]}]
    entree = processor.apply_chat_template(complet, tokenize=True, return_dict=True, return_tensors="pt")
    debut = processor.apply_chat_template(question, add_generation_prompt=True, tokenize=True,
                                          return_dict=True, return_tensors="pt")
    labels = entree["input_ids"].clone()
    labels[:, :debut["input_ids"].shape[1]] = -100
    entree["labels"] = labels
    return entree

def assembler(lot):
    exemples = [preparer(e) for e in lot]
    longueur = max(x["input_ids"].shape[1] for x in exemples)
    pad = processor.tokenizer.pad_token_id
    def completer(t, valeur):
        return torch.nn.functional.pad(t, (0, longueur - t.shape[1]), value=valeur)
    return {
        "input_ids":      torch.cat([completer(x["input_ids"], pad) for x in exemples]),
        "attention_mask": torch.cat([completer(x["attention_mask"], 0) for x in exemples]),
        "labels":         torch.cat([completer(x["labels"], -100) for x in exemples]),
        "pixel_values":   torch.cat([x["pixel_values"] for x in exemples]),
        "image_grid_thw": torch.cat([x["image_grid_thw"] for x in exemples]),
    }

# Contrôle du masque : seule la transcription doit apparaître
lot = assembler(splits["train"][:1])
ids = lot["labels"][0]
print(repr(processor.tokenizer.decode(ids[ids != -100])), "| vrai :", repr(splits["train"][0]["text"]))

## Entraînement

In [ ]:
from transformers import TrainingArguments, Trainer

LOT, ACCUMULATION = (2, 8) if ECHELLE == 1 else (1, 16)    # 16 lignes par mise à jour dans les deux cas
model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(OUT / f"qwen_occitan{SUFFIXE}"),
        per_device_train_batch_size=LOT, per_device_eval_batch_size=LOT,
        gradient_accumulation_steps=ACCUMULATION,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=20,
        num_train_epochs=2, fp16=True, logging_steps=10,
        eval_strategy="steps", eval_steps=100,
        save_strategy="steps", save_steps=100, save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        report_to="none", remove_unused_columns=False, dataloader_num_workers=2,
    ),
    train_dataset=splits["train"], eval_dataset=splits["dev"], data_collator=assembler,
)
t0 = time.time()
trainer.train()
print(f"Durée : {(time.time() - t0) / 3600:.1f} h")
trainer.save_model(str(OUT / f"qwen_occitan_lora{SUFFIXE}"))

## Évaluation sur le test

In [ ]:
import jiwer
from tqdm.auto import tqdm

model.eval()
model.config.use_cache = True

def transcrire(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": CONSIGNE},
    ]}]
    entrees = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                            return_dict=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        sortie = model.generate(**entrees, max_new_tokens=100, do_sample=False)
    return processor.decode(sortie[0, entrees["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def nfd(s):
    return unicodedata.normalize("NFD", s or "").strip()

lignes = [{"image": e["image"], "manuscrit": e["manuscrit"], "vrai": e["text"],
           "pred": transcrire(agrandir(Image.open(chemin_image(e)).convert("RGB")))}
          for e in tqdm(splits["test"])]

with open(OUT / f"predictions_qwen_lora{SUFFIXE}.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["image", "manuscrit", "vrai", "pred"])
    w.writeheader(); w.writerows(lignes)

vrais = [nfd(r["vrai"]) for r in lignes]
preds = [nfd(r["pred"]) for r in lignes]
print(f"CER global : {jiwer.cer(vrais, preds):.1%}")
par_ms = defaultdict(lambda: ([], []))
for r, v, p in zip(lignes, vrais, preds):
    par_ms[r["manuscrit"]][0].append(v); par_ms[r["manuscrit"]][1].append(p)
for m, (v, p) in par_ms.items():
    print(f"  {m:25} {len(v):4} lignes   CER {jiwer.cer(v, p):.1%}   vides : {sum(1 for x in p if not x)}")